# 4 — BankNifty Enhanced Charts V2 (Time-Based X-Axis + Extra Panels)

**Run frequency:** During market hours alongside Notebook 2

**What this does:**
1. Logs in to Zerodha KiteConnect (needed for live ATM strike)
2. Starts a SparkSession
3. Reads instruments + expiries from Parquet
4. Builds the options DataFrame for the configured ATM strike
5. Renders **enhanced V2** Plotly charts with:
   - **Time-based x-axis** (`DD-Mon HH:MM` IST) — no row numbers
   - **Multi-day support**: date+time labels so two-day data doesn't overlap; vertical dashed lines mark day boundaries
   - **Dark theme** with gridlines and unified hover crosshairs
   - **Bollinger Bands** on straddle close
   - **ROC AVG** panel (Premium Decay removed for clarity)
   - **OI PCR** on a secondary Y-axis within the CE/PE OI chart
   - **New chart**: PUT OI − CALL OI (net OI imbalance)
   - **New chart**: Straddle Price + VWAP + OIWAP (OI-Weighted Avg Price)
   - **Volume bars** colour-coded by candle direction
6. **Auto-refreshes** every minute as new Parquet data arrives from Notebook 2

---
### ⚙️ Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| `STRIKE_LEVEL_NAME` | `'ATM'` | Which strike to chart ('ATM', 'ATM+1', 'ATM-2', …) |
| `CE_OR_PE` | `'E'` | `'CE'`, `'PE'`, or `'E'` / `'S'` for straddle |
| `NUM_DAYS` | `2` | Days of data to display |
| `IS_LATEST_DAY` | `True` | Show only today's data |
| `CUSTOM_STRIKE` | `0` | Override ATM (0 = live LTP) |
| `NUM_STRIKES` | `11` | Strikes each side of ATM to load |
| `LOOP_INTERVAL_MIN` | `1` | Chart refresh frequency |

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [ ]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
STRIKE_LEVEL_NAME  = 'ATM'   # 'ATM', 'ATM+1', 'ATM-2', etc.
CE_OR_PE           = 'E'     # 'CE', 'PE', or 'E' for straddle
NUM_DAYS           = 2       # Days of OHLC data to show
IS_LATEST_DAY      = True    # True = show today only; False = show NUM_DAYS
NUM_DAYS_BACK      = 0       # 0 = latest, 1 = go back 1 day
CUSTOM_STRIKE      = 0       # 0 = use live LTP; override e.g. 56500
NUM_STRIKES        = 11      # Strikes each side of ATM to load
LOOP_INTERVAL_MIN  = 1       # Chart refresh frequency (minutes)
INTERVAL           = '3minute'

In [ ]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')

In [ ]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='4_Enhanced_Charts_BNF')
print('✅ Spark session ready')

In [ ]:
# ── Step 3: Load instruments and expiry data ───────────────────────────────
from utils.strike_utils import read_instruments, get_expiry_dates, get_Options_DF, get_ATM_Strike
from utils.strike_utils import BANKNIFTY_INDEX_TOKEN

bnf_options, bnf_futures, nifty_options, nifty_futures, expiries_df = read_instruments(spark)
bnf_expiries = get_expiry_dates(expiries_df, 'BANKNIFTY')

print(f"BankNifty current week expiry: {bnf_expiries['current_week']}")
print(f"BankNifty next week expiry:    {bnf_expiries['next_week']}")

In [ ]:
# ── Step 4: Build options DataFrame for ATM region ────────────────────────
if CUSTOM_STRIKE > 0:
    bnf_atm = CUSTOM_STRIKE
else:
    bnf_atm = get_ATM_Strike(kite, BANKNIFTY_INDEX_TOKEN, rounding_number=100)

print(f'BankNifty ATM Strike = {bnf_atm}')

Banknifty_Options_DF = get_Options_DF(
    spark                = spark,
    options_df_from_file = bnf_options,
    atm_strike           = bnf_atm,
    current_expiry       = bnf_expiries['current_week'],
    strike_range         = 100,
    num_strikes          = NUM_STRIKES,
)
print(f"Options loaded: {Banknifty_Options_DF.count()} rows")
Banknifty_Options_DF.show(5)

In [ ]:
# ── Step 5: Start auto-refreshing ENHANCED V2 charts ──────────────────────
# This cell blocks. Press Kernel → Interrupt to stop.
#
# Charts rendered each refresh:
#   Fig 1 — Price & Signals  (Straddle close, BB bands, SL, AVG, Entry)
#   Fig 2 — ROC AVG          (no Premium Decay)
#   Fig 3 — CE vs PE Close   (straddle)
#   Fig 4 — CE/PE ROC & Ratio (straddle)
#   Fig 5 — Open Interest    (Combined OI + CE/PE OI with PCR on secondary Y)
#   Fig 6 — PUT OI − CALL OI (net OI imbalance)
#   Fig 7 — Straddle Price + VWAP + OIWAP
#   Fig 8 — OHLC Candlestick + Volume
from utils.enhanced_charts_v2 import run_enhanced_chart_loop_v2

run_enhanced_chart_loop_v2(
    spark                 = spark,
    options_df            = Banknifty_Options_DF,
    expiry                = bnf_expiries['current_week'],
    strike_level_name     = STRIKE_LEVEL_NAME,
    ce_or_pe              = CE_OR_PE,
    interval              = INTERVAL,
    num_days              = NUM_DAYS,
    is_latest_day         = IS_LATEST_DAY,
    loop_interval_minutes = LOOP_INTERVAL_MIN,
)